# 2.3 — Méthode par dictionnaire et règles

1. Construire un dictionnaire à partir des mentions annotées de TRAIN.
2. Le chercher dans VALIDATION et régler les choix (casse, filtre, regex).
3. Inspecter 10 sorties, dont des phrases sans entité, puis calculer les scores.
4. Corriger les erreurs restantes avec des règles de post-traitement (suffixes, sigles, contexte « -induced »…).

TEST ne sert jamais à enrichir le dictionnaire ni à construire les règles. Code : `src/dictionnaire.py` (adapté du cours) et règles dans ce notebook ; scores : `src/extraction.py`.

In [1]:
import sys, json, random, re, time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import Mention, erreurs
from tals.spans import tokeniser_avec_offsets
from src.cdr import CATEGORIES, charger_jsonl, mentions_reference, decouper_phrases
from src.dictionnaire import DictionnaireCDR, ConfigDictionnaire, REGEX
from src.extraction import tableau_scores, ecrire_predictions

RESULTATS = RACINE / "resultats" / "extraction"
SEED = 42

documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")  # écrit par 2.2
train = [d for d in documents if d.partition == "train"]
validation = [d for d in documents if d.partition == "validation"]
test = [d for d in documents if d.partition == "test"]
ref_val = mentions_reference(validation)
textes_val = {d.document_id: d.texte for d in validation}
print(len(train), len(validation), len(test), "documents ;", len(ref_val), "mentions de référence en VALIDATION")

500 500 500 documents ; 9591 mentions de référence en VALIDATION


## Construire le dictionnaire à partir de TRAIN

Chaque mention annotée de TRAIN devient une entrée : texte → catégorie. Si un même texte est annoté avec les deux catégories, on garde la plus fréquente.

On calcule aussi le **taux d'annotation** de chaque entrée : quand ce texte apparaît dans TRAIN, à quelle fréquence est-il annoté ? Un taux bas annonce des fausses détections.

In [2]:
base = DictionnaireCDR(ConfigDictionnaire(casse="insensible")).construire(train)
stats = pd.DataFrame([{"forme": " ".join(k), **v} for k, v in base.stats.items()])
print("Entrées :", len(stats))
print(stats["type"].value_counts().to_string())
formes_par_type = {t: {base._cle(m.texte) for d in train for m in d.mentions if m.type == t} for t in CATEGORIES}
ambigues = formes_par_type["Chemical"] & formes_par_type["Disease"]
print("Formes annotées avec les deux types :", len(ambigues), [" ".join(k) for k in sorted(ambigues)][:10])

Entrées : 2390
type
Disease     1384
Chemical    1006
Formes annotées avec les deux types : 1 ['pcp']


In [3]:
# Formes fréquentes dans TRAIN mais rarement annotées : sources probables de faux positifs
stats.query("occurrences >= 5").sort_values("taux").head(15)

,forme,type,annotations,occurrences,taux
1049,mg,Chemical,3,386,0.007772
2133,be,Chemical,7,327,0.021407
564,all,Disease,4,129,0.031008
2151,h,Chemical,3,79,0.037975
471,can,Disease,3,71,0.042254
673,hg,Chemical,1,17,0.058824
1864,acute hepatitis,Disease,1,12,0.083333
221,infarction,Disease,4,48,0.083333
2248,nephritis,Disease,1,11,0.090909
1393,haemorrhage,Disease,1,10,0.100000


## Réglages testés sur VALIDATION

- **Casse** : `exacte` ou `insensible` (« Fentanyl » = « fentanyl »).
- **Filtre** : on retire les entrées dont le taux d'annotation est sous un seuil (0 = pas de filtre).
- **Regex** : terminaisons typiques (Chemical : *-mycin, -cillin, -olol, -pril…* ; Disease : *-itis, -emia, -pathy, -oma…*).
- **Mentions concurrentes** : la plus longue gagne (« renal failure » plutôt que « renal »).

On compare 16 combinaisons.

In [4]:
reglage = []
for casse in ["exacte", "insensible"]:
    for seuil in [0.0, 0.3, 0.5, 0.7]:
        for regex in [False, True]:
            cfg = ConfigDictionnaire(casse, seuil, regex)
            t0 = time.perf_counter()
            dic = DictionnaireCDR(cfg).construire(train)
            pred = dic.extraire_corpus(validation)
            duree = time.perf_counter() - t0
            df = tableau_scores(ref_val, pred, "dictionnaire", "validation", duree).set_index("categorie")
            reglage.append({"casse": casse, "seuil": seuil, "regex": regex, "entrees": len(dic.entrees),
                            "P": df.loc["global (micro)", "precision"], "R": df.loc["global (micro)", "rappel"],
                            "F1": df.loc["global (micro)", "F1"],
                            "F1 Chemical": df.loc["Chemical", "F1"], "F1 Disease": df.loc["Disease", "F1"],
                            "temps_s": round(duree, 1)})
reglage = pd.DataFrame(reglage)
reglage.to_csv(RESULTATS / "dictionnaire_reglage_validation.csv", index=False)
reglage.sort_values("F1", ascending=False)

,casse,seuil,regex,entrees,P,R,F1,F1 Chemical,F1 Disease,temps_s
11,insensible,0.3,True,2348,0.8835,0.6438,0.7449,0.7715,0.7127,2.7
13,insensible,0.5,True,2319,0.8876,0.6340,0.7397,0.7697,0.7029,2.7
1,exacte,0.0,True,2719,0.8675,0.6354,0.7335,0.7554,0.7073,3.0
3,exacte,0.3,True,2683,0.8818,0.6268,0.7328,0.7582,0.7020,3.0
5,exacte,0.5,True,2658,0.8862,0.6180,0.7282,0.7566,0.6934,3.0
10,insensible,0.3,False,2348,0.8903,0.6153,0.7277,0.7557,0.6934,2.5
12,insensible,0.5,False,2319,0.8954,0.6041,0.7215,0.7538,0.6811,2.5
15,insensible,0.7,True,2229,0.8898,0.6055,0.7206,0.7639,0.6663,2.7
0,exacte,0.0,False,2719,0.8690,0.6045,0.7130,0.7352,0.6864,2.9
2,exacte,0.3,False,2683,0.8881,0.5941,0.7119,0.7378,0.6803,2.9


### Les regex sont-elles utiles ?

On compare la meilleure configuration avec et sans regex, et on regarde les mentions ajoutées.

In [5]:
meilleure = reglage.sort_values(["F1", "P"], ascending=False).iloc[0]
cfg_sans = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), False)
cfg_avec = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), True)
sans = DictionnaireCDR(cfg_sans).construire(train).extraire_corpus(validation)
avec = DictionnaireCDR(cfg_avec).construire(train).extraire_corpus(validation)
ajout = avec - sans
print(f"Mentions ajoutées par les regex : {len(ajout)}, dont {len(ajout & ref_val)} correctes "
      f"et {len(ajout - ref_val)} fausses")
for type_ in CATEGORIES:
    ok = Counter(textes_val[m.doc][m.debut:m.fin].lower() for m in ajout & ref_val if m.type == type_)
    ko = Counter(textes_val[m.doc][m.debut:m.fin].lower() for m in ajout - ref_val if m.type == type_)
    print(f"\n{type_} — correctes :", ok.most_common(8))
    print(f"{type_} — fausses   :", ko.most_common(8))

Mentions ajoutées par les regex : 361, dont 274 correctes et 87 fausses

Chemical — correctes : [('fluconazole', 10), ('valsartan', 10), ('netilmicin', 9), ('carteolol', 8), ('ropivacaine', 8), ('levobupivacaine', 7), ('ketoprofen', 7), ('methimazole', 6)]
Chemical — fausses   : [('mitomycin', 8), ('hydroxymetoprolol', 2), ('deoxynojirimycin', 2), ('netilmicin', 1), ('cilastatin', 1), ('piperacillin', 1), ('fenoprofen', 1), ('acylimidazole', 1)]

Disease — correctes : [('hypokalemia', 17), ('uveitis', 11), ('anaemia', 9), ('neuropathy', 8), ('polymyositis', 7), ('arthralgia', 6), ('cytotoxicity', 6), ('thrombophlebitis', 5)]
Disease — fausses   : [('neuropathy', 15), ('carcinoma', 9), ('leukemia', 7), ('rhinitis', 4), ('hyperglycinemia', 4), ('dystrophy', 3), ('lymphoma', 3), ('lysozymuria', 3)]


## Configuration du dictionnaire seul et scores sur VALIDATION

On garde la configuration au meilleur F1 (à égalité, la meilleure précision). Ces scores servent de point de départ aux règles de post-traitement ; les fichiers de la méthode sont écrits après les règles.

In [6]:
CONFIG = ConfigDictionnaire(meilleure["casse"], float(meilleure["seuil"]), bool(meilleure["regex"]))
print("Configuration retenue :", CONFIG.nom())

t0 = time.perf_counter()
dictionnaire = DictionnaireCDR(CONFIG).construire(train)
t_construction = time.perf_counter() - t0
t0 = time.perf_counter()
pred_val = dictionnaire.extraire_corpus(validation)
t_validation = time.perf_counter() - t0
print(f"Construction : {t_construction:.1f} s ; extraction sur VALIDATION : {t_validation:.1f} s")

scores_val = tableau_scores(ref_val, pred_val, "dictionnaire seul", "validation", t_validation)
scores_val

Configuration retenue : casse=insensible, seuil=0.3, regex=oui


Construction : 1.3 s ; extraction sur VALIDATION : 1.2 s


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,dictionnaire seul,validation,Chemical,3501,228,1846,0.9389,0.6548,0.7715,False,NaN
1,dictionnaire seul,validation,Disease,2674,586,1570,0.8202,0.6301,0.7127,False,NaN
2,dictionnaire seul,validation,global (micro),6175,814,3416,0.8835,0.6438,0.7449,False,1.23


## Inspecter dix sorties

5 phrases avec mention et 5 phrases sans mention, tirées au hasard. Notation : `[mention]<Type>` = attendu, `{mention}<Type>` = prédit, `[{mention}]<Type>` = correct.

In [7]:
def marquer(texte, a, b, ref, pred):
    bornes = []
    for m in ref:
        bornes += [(m.debut, 1, "["), (m.fin, 0, f"]<{m.type[:4]}>")]
    for m in pred:
        bornes += [(m.debut, 2, "{"), (m.fin, -1, f"}}<{m.type[:4]}>")]
    sortie, curseur = [], a
    for pos, _, s in sorted(b_ for b_ in bornes if a <= b_[0] <= b):
        sortie += [texte[curseur:pos], s]
        curseur = pos
    sortie.append(texte[curseur:b])
    return "".join(sortie)

phrases_avec, phrases_sans = [], []
for d in validation:
    for a, b in decouper_phrases(d.texte):
        r = {m for m in ref_val if m.doc == d.document_id and a <= m.debut < b}
        p = {m for m in pred_val if m.doc == d.document_id and a <= m.debut < b}
        (phrases_avec if r else phrases_sans).append((d, a, b, r, p))

rng = random.Random(SEED)
for titre, groupe in [("AVEC mention de référence", phrases_avec), ("SANS mention cible", phrases_sans)]:
    print(f"===== 5 phrases {titre} =====")
    for d, a, b, r, p in rng.sample(groupe, 5):
        print(f"[{d.document_id}] VP={len(r & p)} FP={len(p - r)} FN={len(r - p)}")
        print("  ", marquer(d.texte, a, b, r, p))
    print()

fp_sans = sum(len(p) for *_, r, p in phrases_sans)
print(f"Sur les {len(phrases_sans)} phrases sans mention cible de VALIDATION : {fp_sans} fausses détections")

===== 5 phrases AVEC mention de référence =====
[2528969] VP=1 FP=0 FN=0
   Following a 3-week observation period, volunteers were treated with [{zidovudine}<Chem>]<Chem> 600 mg/day for 18 weeks, 900 mg/day for 9 weeks and 1200 mg/day for 9 weeks, followed by a washout period of 6 weeks after which they were re-started on 1200 mg/day or the highest tolerated dose at 8-hourly intervals.
[9022662] VP=1 FP=0 FN=1
   The children had a medical history significant for [attention deficit disorder]<Dise> previously treated with [{methylphenidate}<Chem>]<Chem> without success.
[20705401] VP=3 FP=1 FN=1
   The symptoms of [{methamphetamine}<Chem>]<Chem> ([{METH}<Chem>]<Chem>)-induced [{psychosis}<Dise>]<Dise> are similar to those of [paranoid type {schizophrenia}<Dise>]<Dise>.
[16132524] VP=0 FP=0 FN=1
   Human malignant [brain tumors]<Dise> have a poor prognosis in spite of surgery and radiation therapy.
[16629641] VP=2 FP=2 FN=2
   [Chronic infection with {hepatitis}<Dise> C virus]<Dise> (HCV

## Types d'erreurs sur VALIDATION

- **frontière** : bonne zone, mauvaises limites ;
- **type** : bonnes limites, mauvaise catégorie ;
- **manquée** : mention non trouvée ;
- **parasite** : détection là où il n'y a pas de mention.

Pour les mentions manquées, on vérifie si elles étaient absentes de TRAIN.

In [8]:
err = erreurs(ref_val, pred_val, textes_val)
print({k: len(v) for k, v in err.items() if k != "surfaces"})

inconnues = [m for m in err["manquee"] if dictionnaire._cle(textes_val[m.doc][m.debut:m.fin]) not in dictionnaire.entrees]
print(f"Mentions manquées dont la forme n'est pas dans le dictionnaire : {len(inconnues)} / {len(err['manquee'])}")

def s(m):
    return textes_val[m.doc][m.debut:m.fin]

print("\nExemples de frontière (référence → prédiction) :")
for att, obt in err["frontiere"][:6]:
    print(f"  {s(att)!r}<{att.type}> → {s(obt)!r}<{obt.type}>")
print("\nExemples de confusion de type :")
for att, obt in err["type"][:6]:
    print(f"  {s(att)!r} : référence {att.type}, prédit {obt.type}")
print("\nFormes manquées les plus fréquentes :", Counter(s(m).lower() for m in err["manquee"]).most_common(10))
print("Parasites les plus fréquents :", Counter(s(m).lower() for m in err["parasite"]).most_common(10))

{'correcte': 6175, 'frontiere': 578, 'type': 13, 'manquee': 2825, 'parasite': 223}
Mentions manquées dont la forme n'est pas dans le dictionnaire : 2808 / 2825

Exemples de frontière (référence → prédiction) :
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'secondary hyperparathyroidism'<Disease> → 'hyperparathyroidism'<Disease>
  'ventricular arrhythmia'<Disease> → 'arrhythmia'<Disease>
  'ventricular arrhythmia'<Disease> → 'arrhythmia'<Disease>
  'renal tubular dysfunction'<Disease> → 'tubular dysfunction'<Disease>

Exemples de confusion de type :
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease
  'CM' : référence Chemical, prédit Disease

Formes manquées les plus fréquentes : [('dexmedetomidine', 20), ('no', 1

## Règles de post-traitement

Les erreurs restantes sont surtout des mentions manquées : médicaments absents de TRAIN, sigles, maladies dont il manque un mot à gauche (« rigidity » au lieu de « muscle rigidity »). Cinq règles s'appliquent après le dictionnaire, dans cet ordre :

1. **Suffixes appris** : suffixes de 4 à 6 lettres dont au moins 80 % des mots (≥ 8 occurrences) sont annotés avec un même type dans TRAIN (*-dipine, -anyl, -olam* → Chemical ; *-lemia, -bosis, -zure* → Disease). Ils complètent la liste de suffixes écrite à la main.
2. **Modificateurs à gauche** : mots qui, placés juste avant une maladie du dictionnaire, font partie de l'annotation dans TRAIN (≥ 3 fois et ≥ 70 % des cas : *muscle, tardive, myocardial…*). La mention est étendue vers la gauche.
3. **« X-induced » et « induced by X »** : X est un Chemical (82 % des cas dans TRAIN), sauf mots génériques (*drug, therapy, stress…*) et adjectifs (*-al, -ic, -ive…*).
4. **Sigles définis dans le résumé** : « forme longue (SIGLE) ». Si la forme longue est une mention prédite, toutes les occurrences du sigle dans le document prennent son type ; sinon, le sigle est retiré du document, sauf s'il figure dans le dictionnaire (principe de Schwartz & Hearst, 2003).
5. **Préfixe anti-** : les mots en *anti-* (*antiepileptic, antibiotic, antihypertensive…*) ne sont presque jamais annotés dans TRAIN ; les prédictions qui commencent par « anti » sont retirées.

Une règle n'ajoute une mention que si elle ne chevauche aucune mention déjà retenue. Les sigles passent après les autres règles, pour que leurs formes longues soient déjà détectées.

Les suffixes et les modificateurs sont appris sur TRAIN. La liste d'exclusion de « X-induced » et la condition de retrait des sigles ont été ajustées en regardant les erreurs de VALIDATION : le score VALIDATION des règles est donc légèrement optimiste, seul TEST juge la méthode.

In [9]:
# Règles 1 et 2 : apprises sur TRAIN uniquement
compte = defaultdict(Counter)
for d in train:
    types = {(m.debut, m.fin): m.type for m in d.mentions}
    for r in re.finditer(r"\b[a-z]{7,}\b", d.texte):
        for n in (4, 5, 6):
            compte[r.group()[-n:]][types.get(r.span(), "O")] += 1
SUFFIXES = {}
for suffixe, c in compte.items():
    type_, k = max(((t, c[t]) for t in CATEGORIES), key=lambda x: x[1])
    if sum(c.values()) >= 8 and k / sum(c.values()) >= 0.8:
        SUFFIXES[suffixe] = type_
# On garde le suffixe le plus court quand un plus long donne le même type (« amine » couvre « tamine »)
SUFFIXES = {s: t for s, t in SUFFIXES.items() if not any(o != s and s.endswith(o) and SUFFIXES[o] == t for o in SUFFIXES)}
REGEX_SUFFIXES = {t: re.compile(r"\b[A-Za-z][a-z]{3,}(?:" + "|".join(sorted((s for s, u in SUFFIXES.items() if u == t), key=len, reverse=True)) + r")\b")
                  for t in CATEGORIES}


def mot_avant(texte, pos):
    """Mot qui précède immédiatement la position `pos` : (mot en minuscules, début) ou None."""
    debut = max(0, pos - 40)
    r = re.search(r"([A-Za-z][\w']*)\s+$", texte[debut:pos])
    return (r.group(1).lower(), debut + r.start(1)) if r else None


inclus, exclus = Counter(), Counter()
for d in train:
    maladies = {(m.debut, m.fin) for m in d.mentions if m.type == "Disease"}
    for _, a, b, type_ in dictionnaire._correspondances(d.texte):
        mot = mot_avant(d.texte, a) if type_ == "Disease" else None
        if mot is None:
            continue
        if any(x <= mot[1] and y >= b for x, y in maladies):
            inclus[mot[0]] += 1
        elif (a, b) in maladies:
            exclus[mot[0]] += 1
MODIFICATEURS = {w for w in inclus if inclus[w] >= 3 and inclus[w] / (inclus[w] + exclus[w]) >= 0.7}

for t in CATEGORIES:
    print(f"Suffixes {t} ({sum(u == t for u in SUFFIXES.values())}) :", sorted(s for s, u in SUFFIXES.items() if u == t))
print(f"\nModificateurs à gauche ({len(MODIFICATEURS)}) :", sorted(MODIFICATEURS))

Suffixes Chemical (81) : ['acin', 'aine', 'aline', 'amate', 'amil', 'amine', 'amol', 'amycin', 'anol', 'anyl', 'aride', 'arin', 'atin', 'axel', 'bine', 'chol', 'cium', 'dine', 'done', 'eine', 'enol', 'epam', 'erin', 'erol', 'fene', 'hine', 'hium', 'holine', 'honium', 'icin', 'idol', 'iine', 'iline', 'imus', 'inol', 'irin', 'kinin', 'lcet', 'llin', 'lline', 'lone', 'mide', 'mole', 'ndol', 'nidate', 'ntin', 'oban', 'ocin', 'odopa', 'ofol', 'ohol', 'olam', 'olide', 'olol', 'osin', 'othane', 'otide', 'phen', 'pine', 'pion', 'pril', 'ptan', 'rine', 'rone', 'rtan', 'sine', 'sium', 'sone', 'tasy', 'tine', 'tinine', 'toin', 'tone', 'trogen', 'uine', 'urane', 'xate', 'xine', 'xone', 'zine', 'zole']
Suffixes Disease (27) : ['ache', 'alepsy', 'alic', 'bosis', 'brosis', 'cities', 'enia', 'haly', 'hmia', 'hosis', 'hrosis', 'iasis', 'isia', 'lemia', 'lsions', 'mnesia', 'nemia', 'raine', 'rmia', 'sias', 'toma', 'uising', 'uria', 'xicity', 'xiety', 'zure', 'zures']

Modificateurs à gauche (49) : ['abd

In [10]:
EXCLUS_INDUIT = {"drug", "drugs", "stress", "therapy", "anesthesia", "handling", "blockade", "stimulus", "contrast",
                 "acid", "exercise", "surgery", "treatment", "injury", "ischemia", "hypoxia", "radiation", "diet",
                 "agonist", "antagonist", "neuroleptic", "analgesic", "psychostimulant", "the", "both", "rapid",
                 "maximal", "single", "combined", "chronic", "saline"}
ADJECTIF = re.compile(r".*(?:al|ic|ive|ous|ed|ing|ar)")


def libre(m, P):
    """Vrai si la mention ne chevauche aucune mention de P (P : mentions d'un même document)."""
    return all(m.fin <= x.debut or x.fin <= m.debut for x in P)


def r_suffixes(d, P):
    P = set(P)
    for type_, motif in REGEX_SUFFIXES.items():
        for r in motif.finditer(d.texte):
            m = Mention(d.document_id, r.start(), r.end(), type_)
            if libre(m, P):
                P.add(m)
    return P


def r_modificateurs(d, P):
    P = set(P)
    for m in sorted((m for m in P if m.type == "Disease"), key=lambda m: -m.debut):
        mot = mot_avant(d.texte, m.debut)
        if mot and mot[0] in MODIFICATEURS:
            etendue = Mention(m.doc, mot[1], m.fin, "Disease")
            if libre(etendue, P - {m}):
                P = (P - {m}) | {etendue}
    return P


def r_induit(d, P):
    P = set(P)
    for motif in (r"(?<![\w-])([A-Za-z][A-Za-z0-9]{3,})-induced\b", r"\binduced by (?:the |a )?([A-Za-z][A-Za-z0-9-]{3,})\b"):
        for r in re.finditer(motif, d.texte):
            mot = r.group(1).lower()
            if mot in EXCLUS_INDUIT or ADJECTIF.fullmatch(mot):
                continue
            m = Mention(d.document_id, *r.span(1), "Chemical")
            if libre(m, P):
                P.add(m)
    return P


def r_sigles(d, P):
    t, P = d.texte, set(P)
    for r in re.finditer(r"\s\(([A-Za-z0-9][A-Za-z0-9-]{0,9})\)", t):
        sigle = r.group(1)
        if not re.search(r"[A-Z]", sigle):
            continue
        longues = [m for m in P if m.fin == r.start()]
        occurrences = [(x.start(), x.end()) for x in re.finditer(r"(?<![\w-])" + re.escape(sigle) + r"(?![\w-])", t)]
        if longues:
            type_ = longues[0].type
            P = {m for m in P if not ((m.debut, m.fin) in occurrences and m.type != type_)}
            for a, b in occurrences:
                m = Mention(d.document_id, a, b, type_)
                if libre(m, P):
                    P.add(m)
        elif dictionnaire._cle(sigle) not in dictionnaire.entrees:
            P = {m for m in P if (m.debut, m.fin) not in occurrences}
    return P


def r_anti(d, P):
    return {m for m in P if not d.texte[m.debut:m.fin].lower().startswith("anti")}


REGLES = {"suffixes appris": r_suffixes, "modificateurs à gauche": r_modificateurs, "X-induced": r_induit,
          "sigles définis": r_sigles, "préfixe anti-": r_anti}


def appliquer_regles(pred, documents, regles=tuple(REGLES.values())):
    par_doc = defaultdict(set)
    for m in pred:
        par_doc[m.doc].add(m)
    sortie = set()
    for d in documents:
        P = par_doc[d.document_id]
        for regle in regles:
            P = regle(d, P)
        sortie |= P
    return sortie

### Effet de chaque règle sur VALIDATION

Chaque règle est d'abord appliquée seule au dictionnaire, puis toutes ensemble dans l'ordre ci-dessus. On garde une règle si elle améliore le F1.

In [11]:
def ligne(nom, pred):
    df = tableau_scores(ref_val, pred, "dictionnaire", "validation").set_index("categorie")
    return {"regles": nom, "P": df.loc["global (micro)", "precision"], "R": df.loc["global (micro)", "rappel"],
            "F1": df.loc["global (micro)", "F1"], "F1 Chemical": df.loc["Chemical", "F1"], "F1 Disease": df.loc["Disease", "F1"],
            "ajouts": len(pred - pred_val), "ajouts justes": len((pred - pred_val) & ref_val),
            "retraits": len(pred_val - pred), "retraits justes": len((pred_val - pred) & ref_val)}

lignes = [ligne("aucune (dictionnaire seul)", pred_val)]
for nom, regle in REGLES.items():
    lignes.append(ligne(f"+ {nom}", appliquer_regles(pred_val, validation, [regle])))

t0 = time.perf_counter()
pred_val_regles = appliquer_regles(pred_val, validation)
t_regles_val = time.perf_counter() - t0
lignes.append(ligne("toutes, dans l'ordre", pred_val_regles))

effet_regles = pd.DataFrame(lignes)
effet_regles.to_csv(RESULTATS / "regles_effet_validation.csv", index=False)
print(f"Application des règles sur VALIDATION : {t_regles_val:.1f} s")
effet_regles

Application des règles sur VALIDATION : 0.2 s


,regles,P,R,F1,F1 Chemical,F1 Disease,ajouts,ajouts justes,retraits,retraits justes
0,aucune (dictionnaire seul),0.8835,0.6438,0.7449,0.7715,0.7127,0,0,0,0
1,+ suffixes appris,0.8814,0.6696,0.7610,0.7997,0.7128,297,247,0,0
2,+ modificateurs à gauche,0.8901,0.6486,0.7504,0.7715,0.7249,90,64,90,18
3,+ X-induced,0.8814,0.6495,0.7479,0.7767,0.7127,78,54,0,0
4,+ sigles définis,0.8846,0.6744,0.7653,0.7935,0.7310,328,293,5,0
5,+ préfixe anti-,0.8857,0.6438,0.7456,0.7728,0.7129,0,0,17,0
6,"toutes, dans l'ordre",0.8889,0.7128,0.7912,0.8289,0.7434,813,679,112,18


### Ce que les règles corrigent et ce qu'elles introduisent

Les ajouts faux et les retraits justes montrent les limites de chaque règle.

In [12]:
def s(m):
    return textes_val[m.doc][m.debut:m.fin]

for nom, regle in REGLES.items():
    p = appliquer_regles(pred_val, validation, [regle])
    print(f"--- {nom}")
    if p - pred_val:
        print("  ajouts justes :", Counter(s(m) for m in (p - pred_val) & ref_val).most_common(8))
        print("  ajouts faux   :", Counter(s(m) for m in (p - pred_val) - ref_val).most_common(8))
    if pred_val - p:
        print("  retraits de faux positifs :", Counter(s(m) for m in (pred_val - p) - ref_val).most_common(8))
        print("  retraits de mentions justes :", Counter(s(m) for m in (pred_val - p) & ref_val).most_common(8))

err_regles = erreurs(ref_val, pred_val_regles, textes_val)
print("\nErreurs avant règles :", {k: len(v) for k, v in err.items() if k != "surfaces"})
print("Erreurs après règles :", {k: len(v) for k, v in err_regles.items() if k != "surfaces"})

--- suffixes appris
  ajouts justes : [('amlodipine', 17), ('dexmedetomidine', 14), ('quetiapine', 14), ('fluvoxamine', 11), ('ritanserin', 7), ('Dexmedetomidine', 6), ('tropicamide', 6), ('ziprasidone', 6)]
  ajouts faux   : [('aminopyridine', 5), ('tubocurarine', 3), ('piperazine', 3), ('fluocinolone', 2), ('pentaglutamate', 2), ('epipregnanolone', 2), ('metaiodobenzylguanidine', 2), ('ketopregnenolone', 2)]
--- modificateurs à gauche
  ajouts justes : [('muscle rigidity', 15), ('postoperative myalgia', 6), ('postoperative delirium', 5), ('myocardial hypertrophy', 4), ('ocular hypertension', 4), ('pulmonary toxicity', 3), ('cardiovascular toxicity', 3), ('ventricular arrhythmia', 3)]
  ajouts faux   : [('muscle rigidity', 4), ('cardiovascular thrombotic', 3), ('pulmonary hypertension', 3), ('malignant pain', 2), ('ischemic cardiac disease', 1), ('intracranial hematoma', 1), ('postoperative hallucinations', 1), ('idiopathic dystonia', 1)]
  retraits de faux positifs : [('rigidity', 20

--- sigles définis
  ajouts justes : [('AF', 14), ('NO', 12), ('NTG', 12), ('RIHA', 10), ('CP', 9), ('CYP', 9), ('CIN', 9), ('NIMO', 9)]
  ajouts faux   : [('mRNA', 4), ('P20', 4), ('DLT', 3), ('P45', 3), ('AAS', 2), ('MuLV', 2), ('MMG', 2), ('HIV', 2)]
  retraits de faux positifs : [('MP', 5)]
  retraits de mentions justes : []
--- préfixe anti-
  retraits de faux positifs : [('antidepressant', 11), ('antidepressants', 4), ('antileukemia', 1), ('antigenemia', 1)]
  retraits de mentions justes : []



Erreurs avant règles : {'correcte': 6175, 'frontiere': 578, 'type': 13, 'manquee': 2825, 'parasite': 223}
Erreurs après règles : {'correcte': 6836, 'frontiere': 576, 'type': 8, 'manquee': 2171, 'parasite': 270}


## Scores de la méthode sur VALIDATION (dictionnaire + règles)

Ce sont les scores et les prédictions de la méthode retenue (repris par `2.4_methode_apprise.ipynb` pour la comparaison). Le temps compte l'extraction et les règles.

In [13]:
t_validation_total = t_validation + t_regles_val
scores_val_regles = tableau_scores(ref_val, pred_val_regles, "dictionnaire", "validation", t_validation_total)
scores_val_regles.to_csv(RESULTATS / "scores_dictionnaire_validation.csv", index=False)
ecrire_predictions(pred_val_regles, "dictionnaire", validation, RESULTATS / "predictions_dictionnaire_validation.jsonl")
scores_val_regles

,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,dictionnaire,validation,Chemical,3999,303,1348,0.9296,0.7479,0.8289,False,NaN
1,dictionnaire,validation,Disease,2837,551,1407,0.8374,0.6685,0.7434,False,NaN
2,dictionnaire,validation,global (micro),6836,854,2755,0.8889,0.7128,0.7912,False,1.43


## Gel et prédictions sur TEST

La configuration et les règles sont figées. Le dictionnaire et les règles, construits sur TRAIN, sont appliqués aux textes de TEST ; le score TEST est calculé dans `5_evaluation.ipynb`. On sauvegarde le dictionnaire, la configuration, les règles apprises, les prédictions et les temps.

In [14]:
t0 = time.perf_counter()
pred_test = appliquer_regles(dictionnaire.extraire_corpus(test), test)
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test, "dictionnaire", test, RESULTATS / "predictions_dictionnaire_test.jsonl")

pd.DataFrame([{"forme": " ".join(k), "categorie": t, **{c: v for c, v in dictionnaire.stats[k].items() if c != "type"}}
              for k, t in sorted(dictionnaire.entrees.items())]
             ).to_csv(RESULTATS / "dictionnaire.tsv", sep="\t", index=False)

gel = {"config": vars(CONFIG), "entrees": len(dictionnaire.entrees),
       "regex": {k: v.pattern for k, v in REGEX.items()} if CONFIG.regex else None,
       "regles": {"ordre": list(REGLES),
                  "suffixes_appris": dict(sorted(SUFFIXES.items())),
                  "modificateurs_gauche": sorted(MODIFICATEURS),
                  "exclus_induit": sorted(EXCLUS_INDUIT)},
       "temps_s": {"construction_train": round(t_construction, 2), "extraction_validation": round(t_validation_total, 2),
                   "extraction_test": round(t_test, 2)},
       "predictions_test": len(pred_test)}
(RESULTATS / "dictionnaire_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in gel.items() if k != "regles"}, ensure_ascii=False, indent=2))
print("Règles :", gel["regles"]["ordre"])

{
  "config": {
    "casse": "insensible",
    "seuil_annotation": 0.3,
    "regex": true
  },
  "entrees": 2348,
  "regex": {
    "Chemical": "\\b[A-Za-z][a-z]{2,}(?:mycin|micin|cillin|cycline|azole|olol|pril|sartan|platin|statin|tidine|azepam|azolam|caine|profen|oxacin|vudine|navir|parin|semide|thiazide|triptyline|pramine|peridol|zodone|xetine|barbital)\\b",
    "Disease": "\\b[A-Za-z][a-z]{2,}(?:itis|emia|aemia|pathy|toxicity|algia|plegia|penia|cytosis|trophy|oma|uria)\\b"
  },
  "temps_s": {
    "construction_train": 1.33,
    "extraction_validation": 1.43,
    "extraction_test": 1.72
  },
  "predictions_test": 7995
}
Règles : ['suffixes appris', 'modificateurs à gauche', 'X-induced', 'sigles définis', 'préfixe anti-']
